# How stable are the well-log models across wells and rock types?
Liam Mora — INDE 577

## Investigation contract
Can the apparent sonic-prediction improvement survive different development-well
assignments, and where does it fail? An analyst reconstructing incomplete logs needs
to know whether a pooled score hides unreliable wells or rock types. A secondary
question concerns the stability of sandstone screening at the existing 0.5 threshold.
The observation is a depth measurement; the independent split unit is a whole well.
This notebook summarizes fixed-model sensitivity, not a new tuning search.

## Data provenance and eligibility
The [FORCE 2020 repository](https://github.com/bolgebrygg/Force-2020-Machine-Learning-competition)
and [archived release](https://zenodo.org/records/4351156) supply the training archive
under NLOD 2.0. Verified November 19, 2026, it contains 1,170,511 observations from
98 wells. See `docs/FORCE_DATA_CHECK.md` for acquisition and validation;
`configs/reserved_wells.json` records its checksum and the 20 reserved wells.
The loader checks schema, numerical validity, duplicate well/depth pairs and labels.

Inputs are GR (API), RHOB (g/cm³), and NPHI (porosity fraction). The continuous target
is DTC (µs/ft). Binary sandstone means code 30000; all other observed lithology codes,
including mixed sandstone/shale 65030, are negative. Well identity, depth and rock
labels are never regression predictors. Missing inputs or task targets exclude a row;
missing lithology alone does not exclude an otherwise eligible regression row.
Complete-case evidence does not establish performance where measurements are missing.

## Evaluation boundaries
Seeds 577–581 each assign 58 training and 20 validation wells from 78 development
wells. All 20 final-test wells remain excluded. Assignment precedes eligibility
filtering. At most 250 eligible rows per training well are sampled using seed 577.
Changing well membership may also change the sample within shared training wells.
All candidates within a task and split use identical rows. Scaling and PCA fit only
training data. Validation uses every eligible row rather than a capped sample.

These holdouts overlap and follow earlier development; seed 577 was used for tree
selection. Ranges are descriptive sensitivity, **not confidence intervals or five
independent tests**. We do not combine repeated validation rows into a larger dataset.
Depth dependence persists within wells. No hyperparameter or threshold is retuned.

## Mathematical model and fixed comparisons
For observed sonic slowness y and prediction ŷ, MAE = mean(|ŷ−y|),
RMSE = sqrt(mean((ŷ−y)²)), and signed bias = mean(ŷ−y), all in µs/ft.
Pooled MAE weights wells by eligible row count; mean per-well MAE weights wells equally.
Neither metric removes geographic or geological sampling bias.

The mean and median of sampled training targets provide constant baselines. They
minimize squared and absolute training loss respectively, not validation loss.
Custom linear regression minimizes full mean squared error by batch gradient descent.
The custom regression tree greedily reduces within-leaf squared error and predicts
leaf means, with depth 4 and minimum leaf size 50 fixed from earlier development.
The supporting PCA model projects training-standardized predictors onto two full-SVD
components and fits the same custom linear regression. It optimizes representation
variance before prediction, not geological relevance.

Custom logistic regression minimizes binary cross-entropy and predicts sandstone when
sigmoid(wᵀz+b) ≥ 0.5. Recall = TP/(TP+FN), precision = TP/(TP+FP), and balanced accuracy
is the mean of sensitivity and specificity. Average precision summarizes ranking over
thresholds. A training-prior classifier supplies the reference baseline.

## Implementation and verification
`dsm_project.experiments.force_robustness.run_experiment` orchestrates the existing
custom `LectureLinearRegressor`, `SquaredErrorTreeRegressor`, and
`BatchLogisticClassifier`. Scaling, PCA and dummy baselines are scikit-learn components.
Earlier model notebooks document agreement with trusted implementations. Tests in
`tests/test_force_robustness.py` check reserved-target isolation, training-only
baselines, missing-label slice accounting and duplicate-seed rejection.
The next cell reruns the experiment; no saved metric table is required as input.

In [ ]:
from pathlib import Path
import os
import json
import matplotlib.pyplot as plt
from IPython.display import display
from dsm_project.experiments.force_robustness import run_experiment

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/force.toml").is_file())
os.chdir(ROOT)
result = run_experiment()
OUT = ROOT / "artifacts/force_robustness"
reserved = set(json.loads((ROOT / "configs/reserved_wells.json").read_text(
    encoding="utf-8-sig"))["reserved_test_wells"])
for split in result["split_manifests"]:
    training, validation = set(split["training_wells"]), set(split["validation_wells"])
    assert len(training) == 58 and len(validation) == 20
    assert not training & validation
    assert not (training | validation) & reserved
assert not result["report"]["final_test_scored"]
assert result["report"]["all_converged"]
coverage = result["coverage"].copy()
coverage["validation_retained_fraction"] = (
    coverage.validation_eligible_rows / coverage.validation_partition_rows)
display(coverage.round(3))

The coverage table makes the population change visible: regression and classification
need different observed targets, and their eligible counts differ. The cap balances
training contributions approximately; it does not rebalance validation wells.
Retained fractions quantify exclusions, not whether missingness is random.

## Split-level error and within-well variation
First compare pooled MAE, then plot the individual well errors for the original
development split. Lines connecting split seeds are visual guides, not time trends.
Lower MAE is better; the baseline panel uses a wider range than the model panel.

In [ ]:
regression = result["regression"]
mae = regression.pivot(index="seed", columns="model", values="mae")
rmse = regression.pivot(index="seed", columns="model", values="rmse")
display(mae.round(3))
display(rmse.round(3))
labels = {"mean_baseline": "Training mean", "median_baseline": "Training median",
          "linear": "Linear", "tree": "Tree", "pca2_linear": "PCA2 + linear"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, names in zip(axes, [("mean_baseline", "median_baseline"),
                          ("linear", "tree", "pca2_linear")]):
    for name in names:
        ax.plot(mae.index.astype(str), mae[name], marker="o", label=labels[name])
    ax.set(xlabel="Development split seed", ylabel="Pooled MAE (µs/ft)")
    ax.legend()
    ax.grid(alpha=0.2)
axes[0].set_title("Constant baselines")
axes[1].set_title("Fixed model comparisons")
fig.savefig(OUT / "split_mae.png", dpi=150)
plt.show()

well = result["per_well"]
well_mae = well[well.seed == 577].pivot(index="slice", columns="model", values="mae")
well_mae = well_mae.sort_values("linear")
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for name, marker in [("linear", "o"), ("tree", "x")]:
    ax.scatter(well_mae[name], well_mae.index, label=labels[name], marker=marker)
ax.set(xlabel="Well MAE (µs/ft)", ylabel="Validation well",
       title="Seed 577: pooled improvements can hide individual failures")
ax.legend()
ax.grid(axis="x", alpha=0.2)
fig.savefig(OUT / "well_mae.png", dpi=150)
plt.show()
display(well.groupby(["seed", "model"]).mae.mean().unstack().round(3))

The tree has lower pooled MAE than linear regression in all five splits, but the
advantage ranges from only about 0.07 to 1.22 µs/ft. At seed 579, tree RMSE is worse
(11.83 versus 11.39): improved typical absolute error does not guarantee better
large-error control. Individual wells can also favor linear regression. Equal-weight
well averages complement the pooled scores; they do not provide independent replicates.
PCA2 improves linear MAE in four splits, with modest differences and one reversal.

## Failure analysis: rock-type support and signed error
The following slice table uses only seed 577 and compares models on identical
eligible rows. Counts of rows **and distinct wells** accompany every lithology code.
Codes are supplied audit labels, not explanatory causes or available serving inputs.
RMSE and signed bias are retained so that MAE cannot hide direction or large errors.

In [ ]:
rock = result["rock_type"].query("seed == 577")
support = rock[rock.model == "linear"].set_index("slice")[["rows", "wells"]]
rock_mae = rock.pivot(index="slice", columns="model", values="mae")
display(support.join(rock_mae[["median_baseline", "linear", "tree"]]).round(3))
display(rock[rock.model == "tree"][["slice", "rows", "wells", "rmse", "bias"]].round(3))
plot_data = support.join(rock_mae[["linear", "tree"]]).sort_values("tree")
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
positions = list(range(len(plot_data)))
for name, marker in [("linear", "o"), ("tree", "x")]:
    ax.scatter(plot_data[name], positions, marker=marker, label=labels[name])
ax.set_yticks(positions, [f"{label}: {int(row.rows):,} rows / {int(row.wells)} wells"
                         for label, row in plot_data.iterrows()])
ax.set(xlabel="Slice MAE (µs/ft)", ylabel="Lithology code and support",
       title="Seed 577: error differs across supplied rock labels")
ax.legend()
ax.grid(axis="x", alpha=0.2)
fig.savefig(OUT / "rock_mae.png", dpi=150)
plt.show()

The tree is slightly worse for sandstone (30000), despite its overall MAE advantage.
Code 90000 is a stronger failure: linear and tree MAE both exceed the training-median
baseline (approximately 37.39 and 29.42 versus 17.36 µs/ft). Code 99000 also has high
model error. A pooled improvement therefore cannot justify uniform trust.
Code 70032 represents one well, and code 74000 only 78 rows: small, dependent slices
are fragile evidence. We retain codes rather than guessing geological names.
No rock-specific model, threshold or causal explanation is selected here.

## Sandstone screening stability
The threshold remains 0.5. Display both baseline and model metrics, then compare
recall with precision across splits. A useful ranking score does not make a fixed
threshold operationally acceptable. The final table shows positive flag rates by
true rock label: for sandstone this is recall; for every other label it is a
false-positive rate under the binary task, **not precision**.

In [ ]:
classification = result["classification"]
display(classification[["seed", "model", "rows", "prevalence", "balanced_accuracy",
                        "precision", "recall", "average_precision"]].round(3))
logistic = classification[classification.model == "logistic"].set_index("seed")
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for metric in ("precision", "recall", "balanced_accuracy"):
    ax.plot(logistic.index.astype(str), logistic[metric], marker="o",
            label=metric.replace("_", " ").capitalize())
ax.set(xlabel="Development split seed", ylabel="Fraction", ylim=(0, 1),
       title="Fixed threshold 0.5: recall remains limited")
ax.legend()
ax.grid(alpha=0.2)
fig.savefig(OUT / "classification_stability.png", dpi=150)
plt.show()
display(result["classification_rock_type"].query(
    "seed == 577 and model == 'logistic'").drop(columns=["seed", "model"]).round(3))

Recall ranges from approximately 0.224 to 0.383, so most sandstone observations
are missed at this threshold. Precision varies from about 0.374 to 0.770. The prior
baseline has zero recall and balanced accuracy 0.5. Logistic improvement over that
baseline does not establish safe autonomous interpretation. Mixed sandstone/shale
is negative by definition, making the task narrower than detecting any sandstone.

## Conclusion and production decision
The fixed shallow tree remains the leading sonic candidate by development MAE;
linear regression remains an interpretable comparison. Neither is uniformly best
across wells, rock types or error metrics. This notebook changes no settings and
does not itself authorize deployment. PCA is supporting representation evidence.
The classifier is an exploratory screening aid with explicit low-recall limitations,
not a substitute for geological review or an indicator of oil presence.

These checks describe within-source transfer to other development wells. They cannot
establish performance on a new basin, different measurement units, missing predictors,
or truly unobserved DTC. Physical plausibility, out-of-range inputs, depth dependence,
unequal well support and missingness require explicit handling in the product contract.
No final-test well has been fitted or scored. Next, freeze the model, supported-input
contract and evaluation protocol before evaluating the reserved wells once. Complete
the template's product, monitoring and documentation requirements without expanding
the algorithm scope unnecessarily.

## Reproduce
Use the DSM Project kernel and Run All with the audited local archive present.
The runner saves tables, sample identities, convergence and split manifests under
`artifacts/force_robustness/`; this notebook also saves its four figures there.
Tracked outputs are cleared to follow the notebook standard. Run
`uv run --locked pytest tests/test_force_robustness.py tests/test_robustness_notebook.py -q`
to verify the analysis boundaries and complete notebook execution.